# 03 — Il confronto

Tutti i modelli finiti (feature, baseline, transformer con i loro seed), sugli stessi fold **per gruppo**. Le
metriche principali, fissate prima dei risultati: **F1 macro a 5 classi** e **accuratezza binaria**, per
testata. Si riportano anche accuratezza a 5 classi, tolerant 0.5 e 1, e il dettaglio per livello.

Gli intervalli al 95% vengono da un **bootstrap per gruppo** (2000 ricampionamenti dei gruppi di testate: le
testate dello stesso gruppo sbagliano insieme). Per ogni transformer, la **differenza** con il classificatore a
feature (quello con C scelto in validazione interna, `feature_tuned`) sugli stessi ricampionamenti.

Per le configurazioni con più seed c'è anche l'**ensemble** (`..._ensembleN`): la media delle probabilità dei seed
per ogni articolo, decisa prima di vedere i risultati.

**Versione 2**, decise prima dei risultati:

- la **combinazione** `combinazione_tfidf_modernbert`: per ogni articolo la media, con peso fisso 0,5 e 0,5, delle
  probabilità di `feature_tuned` e dell'ensemble dei 3 seed di ModernBERT-base; nessuna taratura;
- i **controlli** `..._tetto5`: gli stessi modelli addestrati solo sulle testate valutate, come nella prima versione,
  per vedere quanto conta allargare il training;
- in fondo, il confronto con la **prima versione** (`../../Results/2-classificazione`), sulle stesse testate di test
  se la pulizia nuova non le ha cambiate.

Gira da solo alla fine di ogni esperimento della coda: si può guardare anche mentre la coda lavora.

In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from comune import CLASSI, RESULTS, VERSIONI, metriche_di, per_testata, per_testata_media

SCHEMA = "fold_gruppo"
B = 2000

In [2]:
modelli = {}
for f in sorted(RESULTS.glob(f"feature*_{SCHEMA}_articoli.csv")):
    modelli[f.name.replace(f"_{SCHEMA}_articoli.csv", "")] = pd.read_csv(f)
for d in sorted((RESULTS / "transformer").glob("*")):
    if (d / "FATTO").exists():
        modelli[d.name] = pd.read_csv(d / "articoli.csv")
# ensemble dei seed: per ogni configurazione con più seed finiti, la media delle probabilità per articolo
per_config = {}
for nome in list(modelli):
    m = re.match(r"(.+)_s\d+$", nome)
    if m:
        per_config.setdefault(m[1], []).append(nome)
for config, nomi_seed in per_config.items():
    if len(nomi_seed) > 1:
        parti = [modelli[n].set_index("article_id") for n in nomi_seed]
        e = parti[0][["domain", "fold", "y_true"]].copy()
        e[CLASSI] = sum(p[CLASSI] for p in parti) / len(parti)
        e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
        modelli[f"{config}_ensemble{len(nomi_seed)}"] = e.reset_index()
# la combinazione fissata prima dei risultati: media 0,5/0,5 delle probabilità di TF-IDF e ModernBERT-base (ensemble dei seed)
COMBINAZIONE = ("feature_tuned", "modernbert_base_1024_ep2_ensemble3")
if all(c in modelli for c in COMBINAZIONE):
    a, b = (modelli[c].set_index("article_id") for c in COMBINAZIONE)
    e = a[["domain", "fold", "y_true"]].copy()
    e[CLASSI] = 0.5 * a[CLASSI] + 0.5 * b.loc[a.index, CLASSI]
    e["y_pred"] = e[CLASSI].to_numpy().argmax(1)
    modelli["combinazione_tfidf_modernbert"] = e.reset_index()
print("modelli finiti:", list(modelli))
gruppo = pd.read_csv(VERSIONI / "fold_finale.csv").set_index("domain").gruppo

modelli finiti: ['feature', 'feature_tuned', 'feature_tuned_tetto5', 'bge_large_v15', 'e5_large_v2', 'modernbert_base_1024_ep2_s1', 'modernbert_base_1024_ep2_s2', 'modernbert_base_1024_ep2_s42', 'modernbert_base_1024_ep2_tetto5_s42', 'modernbert_base_1024_ep2_ensemble3', 'combinazione_tfidf_modernbert']


In [3]:
PRINCIPALI = ["f1_macro", "bin_accuratezza", "accuratezza", "tolerant_0.5", "tolerant_1", "bin_f1_macro"]
righe, testate = {}, {}
for nome, p in modelli.items():
    for agg, fun in [("voto", per_testata), ("media", per_testata_media)]:
        t = fun(p); testate[(nome, agg)] = t
        righe[(nome, agg)] = {k: v for k, v in metriche_di(t.y_true, t.y_pred).items() if not isinstance(v, list)}
for f in sorted(RESULTS.glob(f"baseline_*_{SCHEMA}_metriche.json")):
    righe[(f.name.split(f"_{SCHEMA}")[0], "-")] = {k: v for k, v in json.load(open(f))["testata_voto"].items() if not isinstance(v, list)}
tutto = pd.DataFrame(righe).T
tutto.index.names = ["modello", "aggregazione"]
tutto.to_csv(RESULTS / "03_confronto_tutte_le_metriche.csv")
tutto[PRINCIPALI].round(3)

f1_macro  bin_accuratezza  \
modello                             aggregazione                              
feature                             voto             0.373            0.750   
                                    media            0.389            0.770   
feature_tuned                       voto             0.370            0.773   
                                    media            0.363            0.763   
feature_tuned_tetto5                voto             0.399            0.734   
                                    media            0.388            0.770   
bge_large_v15                       voto             0.347            0.707   
                                    media            0.347            0.707   
e5_large_v2                         voto             0.322            0.724   
                                    media            0.322            0.724   
modernbert_base_1024_ep2_s1         voto             0.312            0.681   
                                    media            0.323            0.704   
modernbert_base_1024_ep2_s2         voto             0.338            0.691   
                                    media            0.356            0.701   
modernbert_base_1024_ep2_s42        voto             0.327            0.701   
                                    media            0.364            0.740   
modernbert_base_1024_ep2_tetto5_s42 voto             0.332            0.678   
                                    media            0.341            0.694   
modernbert_base_1024_ep2_ensemble3  voto             0.341            0.684   
                                    media            0.341            0.707   
combinazione_tfidf_modernbert       voto             0.361            0.734   
                                    media            0.355            0.753   
baseline_a_caso                     -                0.214            0.510   
baseline_maggioranza                -                0.096            0.553   

                                                  accuratezza  tolerant_0.5  \
modello                             aggregazione                              
feature                             voto                0.434         0.618   
                                    media               0.451         0.632   
feature_tuned                       voto                0.431         0.615   
                                    media               0.431         0.617   
feature_tuned_tetto5                voto                0.441         0.599   
                                    media               0.444         0.622   
bge_large_v15                       voto                0.401         0.599   
                                    media               0.401         0.599   
e5_large_v2                         voto                0.375         0.581   
                                    media               0.375         0.581   
modernbert_base_1024_ep2_s1         voto                0.395         0.566   
                                    media               0.428         0.594   
modernbert_base_1024_ep2_s2         voto                0.401         0.576   
                                    media               0.418         0.586   
modernbert_base_1024_ep2_s42        voto                0.418         0.581   
                                    media               0.454         0.618   
modernbert_base_1024_ep2_tetto5_s42 voto                0.395         0.556   
                                    media               0.411         0.576   
modernbert_base_1024_ep2_ensemble3  voto                0.424         0.581   
                                    media               0.431         0.599   
combinazione_tfidf_modernbert       voto                0.464         0.628   
                                    media               0.457         0.637   
baseline_a_caso                     -                   0.243         0.388   
baseline_maggioranza

## Seed: media e deviazione standard

Gli esperimenti con lo stesso nome a parte il seed (`..._s42`, `..._s1`, `..._s2`).

In [4]:
x = tutto[PRINCIPALI].reset_index()
x["config"] = x.modello.str.replace(r"_s\d+$", "", regex=True)
seed = x.groupby(["config", "aggregazione"])[PRINCIPALI].agg(["mean", "std", "count"])
seed.to_csv(RESULTS / "03_confronto_seed.csv")
seed.round(3)

f1_macro               \
                                                    mean    std count   
config                             aggregazione                         
baseline_a_caso                    -               0.214    NaN     1   
baseline_maggioranza               -               0.096    NaN     1   
bge_large_v15                      media           0.347    NaN     1   
                                   voto            0.347    NaN     1   
combinazione_tfidf_modernbert      media           0.355    NaN     1   
                                   voto            0.361    NaN     1   
e5_large_v2                        media           0.322    NaN     1   
                                   voto            0.322    NaN     1   
feature                            media           0.389    NaN     1   
                                   voto            0.373    NaN     1   
feature_tuned                      media           0.363    NaN     1   
                                   voto            0.370    NaN     1   
feature_tuned_tetto5               media           0.388    NaN     1   
                                   voto            0.399    NaN     1   
modernbert_base_1024_ep2           media           0.348  0.022     3   
                                   voto            0.326  0.013     3   
modernbert_base_1024_ep2_ensemble3 media           0.341    NaN     1   
                                   voto            0.341    NaN     1   
modernbert_base_1024_ep2_tetto5    media           0.341    NaN     1   
                                   voto            0.332    NaN     1   

                                                bin_accuratezza               \
                                                           mean    std count   
config                             aggregazione                                
baseline_a_caso                    -                      0.510    NaN     1   
baseline_maggioranza               -                      0.553    NaN     1   
bge_large_v15                      media                  0.707    NaN     1   
                                   voto                   0.707    NaN     1   
combinazione_tfidf_modernbert      media                  0.753    NaN     1   
                                   voto                   0.734    NaN     1   
e5_large_v2                        media                  0.724    NaN     1   
                                   voto                   0.724    NaN     1   
feature                            media                  0.770    NaN     1   
                                   voto                   0.750    NaN     1   
feature_tuned                      media                  0.763    NaN     1   
                                   voto                   0.773    NaN     1   
feature_tuned_tetto5               media                  0.770    NaN     1   
                                   voto                   0.734    NaN     1   
modernbert_base_1024_ep2           media                  0.715  0.022     3   
                                   voto                   0.691  0.010     3   
modernbert_base_1024_ep2_ensemble3 media                  0.707    NaN     1   
                                   voto                   0.684    NaN     1   
modernbert_base_1024_ep2_tetto5    media                  0.694    NaN     1   
                                   voto                   0.678    NaN     1   

                                                accuratezza               \
                                                       mean    std count   
config                             aggregazione                            
baseline_a_caso                    -                  0.243    NaN     1   
baseline_maggioranza               -                  0.316    NaN     1   
bge_large_v15                      media              0.401    NaN     1   
                                   voto               0.401    NaN     

## Intervalli e differenze con il feature (bootstrap per gruppo)

In [5]:
rif = ("feature_tuned", "voto") if ("feature_tuned", "voto") in testate else ("feature", "voto")   # il feature al suo meglio, se c'è
nomi = [k for k in testate if k[1] == "media" or k == rif]
dom = testate[rif].index
g = gruppo.reindex(dom).to_numpy()
gruppi = np.unique(g)
pos = {x: np.where(g == x)[0] for x in gruppi}
rng = np.random.default_rng(0)
campioni = [np.concatenate([pos[x] for x in rng.choice(gruppi, len(gruppi))]) for _ in range(B)]


def due(t, idx):
    m = metriche_di(t.y_true.to_numpy()[idx], t.y_pred.to_numpy()[idx])
    return m["f1_macro"], m["bin_accuratezza"]


boot = {k: np.array([due(testate[k].reindex(dom), idx) for idx in campioni]) for k in nomi}
righe = {}
for k in nomi:
    b = boot[k]
    r = {"F1 macro": tutto.loc[k, "f1_macro"], "F1 2.5%": np.percentile(b[:, 0], 2.5), "F1 97.5%": np.percentile(b[:, 0], 97.5),
         "binario": tutto.loc[k, "bin_accuratezza"], "bin 2.5%": np.percentile(b[:, 1], 2.5), "bin 97.5%": np.percentile(b[:, 1], 97.5)}
    if k != rif:
        d = b - boot[rif]
        r.update({"ΔF1 vs feature": tutto.loc[k, "f1_macro"] - tutto.loc[rif, "f1_macro"], "ΔF1 2.5%": np.percentile(d[:, 0], 2.5),
                  "ΔF1 97.5%": np.percentile(d[:, 0], 97.5), "P(ΔF1>0)": (d[:, 0] > 0).mean(),
                  "Δbin vs feature": tutto.loc[k, "bin_accuratezza"] - tutto.loc[rif, "bin_accuratezza"],
                  "Δbin 2.5%": np.percentile(d[:, 1], 2.5), "Δbin 97.5%": np.percentile(d[:, 1], 97.5), "P(Δbin>0)": (d[:, 1] > 0).mean()})
    righe[f"{k[0]} ({k[1]})"] = r
intervalli = pd.DataFrame(righe).T
intervalli.to_csv(RESULTS / "03_confronto_intervalli.csv")
intervalli.round(3)

,F1 macro,F1 2.5%,F1 97.5%,binario,bin 2.5%,bin 97.5%,ΔF1 vs feature,ΔF1 2.5%,ΔF1 97.5%,P(ΔF1>0),Δbin vs feature,Δbin 2.5%,Δbin 97.5%,P(Δbin>0)
feature (media),0.389,0.319,0.456,0.770,0.709,0.829,0.019,-0.005,0.044,0.936,-0.003,-0.024,0.017,0.308
feature_tuned (voto),0.370,0.305,0.434,0.773,0.717,0.828,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
feature_tuned (media),0.363,0.300,0.420,0.763,0.701,0.822,-0.007,-0.037,0.020,0.356,-0.010,-0.034,0.010,0.141
feature_tuned_tetto5 (media),0.388,0.321,0.443,0.770,0.708,0.826,0.017,-0.025,0.064,0.775,-0.003,-0.039,0.032,0.380
bge_large_v15 (media),0.347,0.290,0.400,0.707,0.645,0.768,-0.023,-0.092,0.042,0.270,-0.066,-0.123,-0.007,0.006
e5_large_v2 (media),0.322,0.264,0.376,0.724,0.653,0.788,-0.049,-0.115,0.014,0.072,-0.049,-0.114,0.010,0.047
modernbert_base_1024_ep2_s1 (media),0.323,0.266,0.373,0.704,0.631,0.772,-0.048,-0.114,0.014,0.068,-0.069,-0.124,-0.019,0.002
modernbert_base_1024_ep2_s2 (media),0.356,0.291,0.415,0.701,0.634,0.764,-0.014,-0.089,0.052,0.354,-0.072,-0.133,-0.014,0.003
modernbert_base_1024_ep2_s42 (media),0.364,0.306,0.416,0.740,0.675,0.802,-0.006,-0.078,0.065,0.452,-0.033,-0.082,0.016,0.080
modernbert_base_1024_ep2_tetto5_s42 (media),0.341,0.281,0.400,0.694,0.622,0.762,-0.029,-0.096,0.035,0.203,-0.079,-0.140,-0.023,0.003


## Per livello e matrici di confusione (testata, media delle probabilità; feature col voto)

In [6]:
LIV = ["l1", "l2", "l3", "l4", "l5"]
per_livello = pd.DataFrame({f"{k[0]} ({k[1]})": {f"{l} {q}": tutto.loc[k, f"{l}_{q}"] for l in LIV for q in ["recall", "f1"]} for k in nomi}).T
display(per_livello.round(2))
for k in nomi:
    t = testate[k]
    print(f"\n{k[0]} ({k[1]})")
    print(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]).to_string())

,l1 recall,l1 f1,l2 recall,l2 f1,l3 recall,l3 f1,l4 recall,l4 f1,l5 recall,l5 f1
feature (media),0.68,0.64,0.45,0.42,0.14,0.16,0.50,0.48,0.19,0.24
feature_tuned (voto),0.65,0.61,0.45,0.42,0.14,0.15,0.47,0.46,0.17,0.21
feature_tuned (media),0.66,0.63,0.45,0.42,0.08,0.09,0.48,0.46,0.17,0.20
feature_tuned_tetto5 (media),0.66,0.64,0.48,0.44,0.11,0.14,0.46,0.45,0.25,0.27
bge_large_v15 (media),0.58,0.60,0.48,0.44,0.19,0.18,0.42,0.42,0.08,0.10
e5_large_v2 (media),0.58,0.55,0.37,0.38,0.14,0.13,0.42,0.44,0.11,0.11
modernbert_base_1024_ep2_s1 (media),0.61,0.57,0.54,0.42,0.03,0.04,0.51,0.50,0.06,0.08
modernbert_base_1024_ep2_s2 (media),0.65,0.58,0.55,0.45,0.14,0.17,0.36,0.40,0.14,0.18
modernbert_base_1024_ep2_s42 (media),0.62,0.59,0.58,0.50,0.03,0.04,0.52,0.50,0.14,0.19
modernbert_base_1024_ep2_tetto5_s42 (media),0.59,0.52,0.48,0.41,0.06,0.09,0.45,0.46,0.19,0.23



feature (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        48  12   2   9   0
l2        19  29   6  10   1
l3         6   8   5  16   1
l4         3  18  14  48  13
l5         2   5   0  22   7

feature_tuned (voto)
predetto  l1  l2  l3  l4  l5
vero                        
l1        46  13   2  10   0
l2        21  29   5   9   1
l3         7   6   5  17   1
l4         3  18  16  45  14
l5         2   6   2  20   6

feature_tuned (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        47  12   2  10   0
l2        19  29   6  10   1
l3         7   9   3  16   1
l4         3  16  16  46  15
l5         2   6   2  20   6

feature_tuned_tetto5 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        47  13   3   8   0
l2        18  31   2  12   2
l3         7   8   4  16   1
l4         3  18  13  44  18
l5         2   5   0  20   9

bge_large_v15 (media)
predetto  l1  l2  l3  l4  l5
vero                        
l1        41  1

## Versione 1 contro Versione 2

Le stesse metriche per testata (feature col voto, gli altri con la media delle probabilità). La Versione 1 è in
`../../Results/2-classificazione/03_confronto_tutte_le_metriche.csv`. Le testate di test sono le stesse solo se la
pulizia nuova non ha cambiato l'insieme delle testate valutate: il numero `n` lo dice.

In [7]:
V1 = Path("../../Results/2-classificazione/03_confronto_tutte_le_metriche.csv")
if V1.exists():
    v1 = pd.read_csv(V1, index_col=[0, 1])
    scelte = [k for k in tutto.index if k[1] in ("media", "-") or k == rif]
    confronto = pd.DataFrame({
        f"{k[0]} ({k[1]})": {**{f"v2 {m}": tutto.loc[k, m] for m in ["n", "f1_macro", "bin_accuratezza", "accuratezza"]},
                             **({f"v1 {m}": v1.loc[k, m] for m in ["n", "f1_macro", "bin_accuratezza", "accuratezza"]} if k in v1.index else {})}
        for k in scelte}).T
    confronto.to_csv(RESULTS / "03_confronto_versione1.csv")
    display(confronto.round(3))

,v2 n,v2 f1_macro,v2 bin_accuratezza,v2 accuratezza,v1 n,v1 f1_macro,v1 bin_accuratezza,v1 accuratezza
feature (media),304.0,0.389,0.770,0.451,304.0,0.368,0.776,0.454
feature_tuned (voto),304.0,0.370,0.773,0.431,304.0,0.380,0.740,0.424
feature_tuned (media),304.0,0.363,0.763,0.431,304.0,0.367,0.760,0.421
feature_tuned_tetto5 (media),304.0,0.388,0.770,0.444,NaN,NaN,NaN,NaN
bge_large_v15 (media),304.0,0.347,0.707,0.401,304.0,0.325,0.730,0.382
e5_large_v2 (media),304.0,0.322,0.724,0.375,304.0,0.331,0.727,0.391
modernbert_base_1024_ep2_s1 (media),304.0,0.323,0.704,0.428,304.0,0.348,0.730,0.378
modernbert_base_1024_ep2_s2 (media),304.0,0.356,0.701,0.418,304.0,0.335,0.717,0.365
modernbert_base_1024_ep2_s42 (media),304.0,0.364,0.740,0.454,304.0,0.383,0.720,0.424
modernbert_base_1024_ep2_tetto5_s42 (media),304.0,0.341,0.694,0.411,NaN,NaN,NaN,NaN
